# IFRS S1/S2 Reference Report Style Extraction Pipeline

This notebook extracts **reusable style artifacts** from a reference IFRS S1/S2 report.

It is designed for your project structure:

```text
/notebooks/
  01_style_extraction_from_reference_report.ipynb
  gen_data/
    style/
      emirates_nbd_group_2024_ifrs_s1_s2.pdf
```

Outputs are written under the same directory as the input reference report:

```text
/notebooks/gen_data/style/style_system/
```

Important rule: the reference report is used only for **style, structure, formatting and PDF layout inspiration**. It must not be used as a factual source during report generation.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# Notebook location : /notebooks
# Input report      : /notebooks/gen_data/style/
# Outputs           : /notebooks/gen_data/style/style_system/
# ============================================================

import os
import re
import json
import time
import random
import urllib.request
import urllib.error
import http.client
from pathlib import Path

import pandas as pd

try:
    import fitz  # PyMuPDF
except ImportError:
    raise ImportError("Install PyMuPDF first: pip install pymupdf")

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    raise ImportError("Install python-dotenv first: pip install python-dotenv")


# ------------------------------------------------------------
# 1. Resolve notebook/project paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

# Robust handling:
# - if the notebook is launched from /notebooks, use that
# - if launched from project root, use project_root/notebooks
# - otherwise, fall back to current directory
if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

STYLE_DATA_DIR = NOTEBOOK_DIR / "gen_data" / "style"

if not STYLE_DATA_DIR.exists():
    raise FileNotFoundError(
        "Style data directory not found.\n"
        f"Expected path: {STYLE_DATA_DIR}\n"
        "Create this folder and place the reference report inside it."
    )


# ------------------------------------------------------------
# 2. Input reference report
# ------------------------------------------------------------

DEFAULT_REFERENCE_REPORT_NAME = "emirates_nbd_group_2024_ifrs_s1_s2.pdf"

reference_report_env = os.getenv("REFERENCE_REPORT_PDF")

if reference_report_env:
    candidate = Path(reference_report_env)

    if candidate.is_absolute():
        REFERENCE_REPORT_PDF = candidate
    else:
        # First try relative to the style data folder
        if (STYLE_DATA_DIR / candidate).exists():
            REFERENCE_REPORT_PDF = STYLE_DATA_DIR / candidate
        # Then try relative to notebook directory
        elif (NOTEBOOK_DIR / candidate).exists():
            REFERENCE_REPORT_PDF = NOTEBOOK_DIR / candidate
        # Then try relative to current working directory
        else:
            REFERENCE_REPORT_PDF = CURRENT_DIR / candidate
else:
    REFERENCE_REPORT_PDF = STYLE_DATA_DIR / DEFAULT_REFERENCE_REPORT_NAME

REFERENCE_REPORT_PDF = REFERENCE_REPORT_PDF.resolve()

if not REFERENCE_REPORT_PDF.exists():
    raise FileNotFoundError(
        f"Reference report not found: {REFERENCE_REPORT_PDF}\n\n"
        "Expected default location:\n"
        f"{STYLE_DATA_DIR / DEFAULT_REFERENCE_REPORT_NAME}\n\n"
        "Either place the PDF there, or set REFERENCE_REPORT_PDF in your .env."
    )


# ------------------------------------------------------------
# 3. Output folders
# ------------------------------------------------------------

# Outputs are placed under the same directory as the input report.
STYLE_OUTPUT_DIR = REFERENCE_REPORT_PDF.parent / "style_system"

SECTION_STYLE_DIR = STYLE_OUTPUT_DIR / "section_style_guides"
SECTION_BLUEPRINT_DIR = STYLE_OUTPUT_DIR / "section_blueprints"
TABLE_PATTERN_DIR = STYLE_OUTPUT_DIR / "table_patterns"
LANGUAGE_RULES_DIR = STYLE_OUTPUT_DIR / "language_rules"
INTERMEDIATE_DIR = STYLE_OUTPUT_DIR / "_intermediate"
STYLE_NOTE_CACHE_DIR = INTERMEDIATE_DIR / "style_chunk_notes"

for folder in [
    STYLE_OUTPUT_DIR,
    SECTION_STYLE_DIR,
    SECTION_BLUEPRINT_DIR,
    TABLE_PATTERN_DIR,
    LANGUAGE_RULES_DIR,
    INTERMEDIATE_DIR,
    STYLE_NOTE_CACHE_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 4. Target sections
# ------------------------------------------------------------

TARGET_SECTIONS = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

SECTION_SLUGS = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}


# ------------------------------------------------------------
# 5. Runtime controls
# ------------------------------------------------------------

STYLE_MAX_OUTPUT_TOKENS = int(os.getenv("STYLE_MAX_OUTPUT_TOKENS", "8000"))
STYLE_CHUNK_MAX_CHARS = int(os.getenv("STYLE_CHUNK_MAX_CHARS", "10000"))
STYLE_INTER_REQUEST_DELAY_SECONDS = float(os.getenv("STYLE_INTER_REQUEST_DELAY_SECONDS", "1.0"))
FORCE_STYLE_EXTRACTION = os.getenv("FORCE_STYLE_EXTRACTION", "false").strip().lower() in {"1", "true", "yes", "y"}


# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------

print("Current working directory:", CURRENT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Style data directory:", STYLE_DATA_DIR)
print("Reference report:", REFERENCE_REPORT_PDF)
print("Style output folder:", STYLE_OUTPUT_DIR)
print("Force re-run:", FORCE_STYLE_EXTRACTION)

## 2. Extract PDF text

This reads the reference report page by page using PyMuPDF and saves a page-level text extract for traceability.

In [ ]:
# ============================================================
# CELL 2 — PDF TEXT EXTRACTION
# ============================================================

def clean_pdf_text(text: str) -> str:
    if not text:
        return ""

    # Normalize strange whitespace and PDF control characters.
    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("\u0008", " ")
    text = text.replace("\ufffe", "-")
    text = text.replace("\x08", " ")

    # Collapse spacing but preserve paragraph breaks.
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def extract_pdf_pages(pdf_path: Path) -> pd.DataFrame:
    doc = fitz.open(str(pdf_path))
    rows = []

    for page_index, page in enumerate(doc):
        page_number = page_index + 1
        text = clean_pdf_text(page.get_text("text"))

        rows.append({
            "page": page_number,
            "text": text,
            "char_count": len(text),
        })

    return pd.DataFrame(rows)


pages_df = extract_pdf_pages(REFERENCE_REPORT_PDF)

pages_extract_path = INTERMEDIATE_DIR / "reference_report_pages.csv"
pages_df.to_csv(pages_extract_path, index=False, encoding="utf-8-sig")

print("Pages extracted:", len(pages_df))
print("Saved page extract:", pages_extract_path)
display(pages_df.head())

## 3. Segment the reference report into target sections

The reference report follows the same major section structure needed for your final IFRS S1/S2 report.

This cell uses the visible section boundaries from the reference report contents and divider pages. This is acceptable for style extraction because the reference report is a fixed style source; it is not used for IFRS requirement extraction.

In [ ]:
# ============================================================
# CELL 3 — SECTION SEGMENTATION
# ============================================================

# Section boundaries from the reference report structure.
# If you replace the reference report later, update this dictionary or set
# REFERENCE_SECTION_STARTS_JSON in .env.
# Example .env:
# REFERENCE_SECTION_STARTS_JSON={"General Requirements":4,"Governance":8,"Strategy":19,"Risk Management":47,"Metrics and Targets":55,"Appendix":64}

REFERENCE_SECTION_STARTS_DEFAULT = {
    "General Requirements": 4,
    "Governance": 8,
    "Strategy": 19,
    "Risk Management": 47,
    "Metrics and Targets": 55,
    "Appendix": 64,
}

starts_env = os.getenv("REFERENCE_SECTION_STARTS_JSON")
if starts_env:
    try:
        REFERENCE_SECTION_STARTS = json.loads(starts_env)
        REFERENCE_SECTION_STARTS = {str(k): int(v) for k, v in REFERENCE_SECTION_STARTS.items()}
        print("Loaded section starts from REFERENCE_SECTION_STARTS_JSON")
    except Exception as exc:
        raise ValueError("Invalid REFERENCE_SECTION_STARTS_JSON. It must be valid JSON with page numbers.") from exc
else:
    REFERENCE_SECTION_STARTS = REFERENCE_SECTION_STARTS_DEFAULT


def build_section_ranges(section_starts: dict, max_page: int) -> dict:
    ordered = sorted(section_starts.items(), key=lambda x: x[1])
    ranges = {}

    for i, (section, start_page) in enumerate(ordered):
        if section.lower().startswith("appendix"):
            continue

        next_start = ordered[i + 1][1] if i + 1 < len(ordered) else max_page + 1
        end_page = next_start - 1

        if section in TARGET_SECTIONS:
            ranges[section] = {
                "start_page": int(start_page),
                "end_page": int(end_page),
            }

    missing = [s for s in TARGET_SECTIONS if s not in ranges]
    if missing:
        raise ValueError(f"Missing section ranges for: {missing}")

    return ranges


section_ranges = build_section_ranges(
    REFERENCE_SECTION_STARTS,
    max_page=int(pages_df["page"].max()),
)

print(json.dumps(section_ranges, indent=2))


def get_section_text(section_name: str) -> str:
    start = section_ranges[section_name]["start_page"]
    end = section_ranges[section_name]["end_page"]

    subset = pages_df[
        (pages_df["page"] >= start) &
        (pages_df["page"] <= end)
    ].copy()

    joined = "\n\n".join(
        f"[PAGE {int(row.page)}]\n{row.text}"
        for _, row in subset.iterrows()
        if str(row.text).strip()
    )

    return clean_pdf_text(joined)


section_texts = {
    section: get_section_text(section)
    for section in TARGET_SECTIONS
}

section_text_records = []
for section, text in section_texts.items():
    print(section, "chars:", len(text))
    section_text_records.append({
        "section": section,
        "start_page": section_ranges[section]["start_page"],
        "end_page": section_ranges[section]["end_page"],
        "char_count": len(text),
    })

section_ranges_df = pd.DataFrame(section_text_records)
section_ranges_path = INTERMEDIATE_DIR / "reference_section_ranges.csv"
section_ranges_df.to_csv(section_ranges_path, index=False, encoding="utf-8-sig")

print("Saved section ranges:", section_ranges_path)
display(section_ranges_df)

## 4. Configure Azure GPT-5.2

This notebook uses the same simple full deployment URL style you use elsewhere:

```env
AZURE_OPENAI_API_KEY=...
AZURE_OPENAI_GPT52_DEPLOYMENT_URL=https://.../openai/deployments/.../chat/completions?api-version=...
```

It also accepts `AZURE_OPENAI_STYLE_URL`, `AZURE_OPENAI_EXTRACTOR_URL`, `AZURE_OPENAI_JUDGE_URL`, or `AZURE_OPENAI_CHAT_URL` as fallbacks.

In [ ]:
# ============================================================
# CELL 4 — AZURE OPENAI REST HELPER
# ============================================================

env_path = find_dotenv()
if env_path:
    load_dotenv(env_path, override=True)
    print("Loaded .env from:", env_path)
else:
    load_dotenv(override=True)
    print("No .env found by find_dotenv(); using existing environment variables.")


def _clean_url(value):
    if not value:
        return None
    return str(value).strip().strip('"').strip("'")


AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_STYLE_API_KEY")
    or os.getenv("AZURE_OPENAI_EXTRACTOR_API_KEY")
    or os.getenv("AZURE_OPENAI_JUDGE_API_KEY")
)

AZURE_OPENAI_STYLE_URL = _clean_url(
    os.getenv("AZURE_OPENAI_STYLE_URL")
    or os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
    or os.getenv("AZURE_OPENAI_EXTRACTOR_URL")
    or os.getenv("AZURE_OPENAI_JUDGE_URL")
    or os.getenv("AZURE_OPENAI_CHAT_URL")
)


def validate_style_azure_config():
    missing = []

    if not AZURE_OPENAI_API_KEY:
        missing.append("AZURE_OPENAI_API_KEY")

    if not AZURE_OPENAI_STYLE_URL:
        missing.append(
            "AZURE_OPENAI_STYLE_URL or AZURE_OPENAI_GPT52_DEPLOYMENT_URL or AZURE_OPENAI_JUDGE_URL"
        )

    if missing:
        flags = {
            "AZURE_OPENAI_API_KEY_loaded": bool(os.getenv("AZURE_OPENAI_API_KEY")),
            "AZURE_OPENAI_STYLE_URL_loaded": bool(os.getenv("AZURE_OPENAI_STYLE_URL")),
            "AZURE_OPENAI_GPT52_DEPLOYMENT_URL_loaded": bool(os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")),
            "AZURE_OPENAI_EXTRACTOR_URL_loaded": bool(os.getenv("AZURE_OPENAI_EXTRACTOR_URL")),
            "AZURE_OPENAI_JUDGE_URL_loaded": bool(os.getenv("AZURE_OPENAI_JUDGE_URL")),
            "AZURE_OPENAI_CHAT_URL_loaded": bool(os.getenv("AZURE_OPENAI_CHAT_URL")),
        }
        raise ValueError(
            "Missing Azure style extraction configuration: "
            + ", ".join(missing)
            + "\n\nLoaded flags, keys are never printed:\n"
            + json.dumps(flags, indent=2)
            + "\n\nExpected .env example:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full GPT-5.2 chat-completions deployment URL>"
        )

    if not AZURE_OPENAI_STYLE_URL.startswith("https://"):
        raise ValueError(
            "Azure URL must be a full HTTPS endpoint. "
            f"Current value: {AZURE_OPENAI_STYLE_URL!r}"
        )

    if "/chat/completions" not in AZURE_OPENAI_STYLE_URL:
        print("WARNING: The Azure URL does not contain '/chat/completions'.")
        print("Expected full URL format:")
        print("https://<resource>.openai.azure.com/openai/deployments/<deployment>/chat/completions?api-version=<version>")

    print("Azure style extraction config loaded.")
    print("Endpoint:", AZURE_OPENAI_STYLE_URL[:120] + "...")
    print("API key loaded:", bool(AZURE_OPENAI_API_KEY))


validate_style_azure_config()


def _extract_message_content(data: dict) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except Exception as exc:
        raise ValueError(
            "Unexpected Azure response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure returned empty content:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _extract_json_object(text: str) -> str:
    text = str(text).strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)

    first = text.find("{")
    last = text.rfind("}")

    if first >= 0 and last > first:
        return text[first:last + 1]

    return text


def azure_chat_json(
    *,
    system_prompt: str,
    user_prompt: str,
    request_label: str,
    max_output_tokens: int = STYLE_MAX_OUTPUT_TOKENS,
    timeout: int = 240,
    max_attempts: int = 5,
) -> dict:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Behaviour:
    - Uses the full deployment URL as-is.
    - Retries transient 429/500/502/503/504 and connection errors.
    - For GPT-5.x gateways, first tries max_completion_tokens.
    - If rejected, retries with max_tokens.
    - Requests JSON output and parses a JSON object.
    """
    token_fields = ["max_completion_tokens", "max_tokens"]
    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            token_field: max_output_tokens,
            "temperature": 0.0,
            "response_format": {"type": "json_object"},
        }

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                AZURE_OPENAI_STYLE_URL,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": AZURE_OPENAI_API_KEY,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    data = json.loads(resp.read().decode("utf-8"))

                content = _extract_message_content(data)
                candidate = _extract_json_object(content)
                return json.loads(candidate)

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Token field used: {token_field}\n"
                    f"Response preview: {body[:2000]}"
                )

                transient = exc.code in {429, 500, 502, 503, 504}
                compatibility = token_field == "max_completion_tokens" and exc.code in {400, 422, 500}

                if transient and attempt < max_attempts:
                    retry_after = exc.headers.get("Retry-After")
                    try:
                        wait = float(retry_after) if retry_after else min(2 ** attempt, 30)
                    except ValueError:
                        wait = min(2 ** attempt, 30)

                    wait = min(max(wait, 1.0), 60.0)
                    print(f"{request_label}: HTTP {exc.code}; retrying in {wait:.1f}s...")
                    time.sleep(wait)
                    continue

                if compatibility:
                    print(f"{request_label}: retrying with max_tokens instead of max_completion_tokens.")
                    break

                raise last_error from exc

            except (
                urllib.error.URLError,
                ConnectionResetError,
                TimeoutError,
                OSError,
                http.client.RemoteDisconnected,
                json.JSONDecodeError,
            ) as exc:
                last_error = RuntimeError(
                    f"{request_label} failed on attempt {attempt}/{max_attempts}: {repr(exc)}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** attempt + random.random(), 30)
                    print(f"{request_label}: transient error; retrying in {wait:.1f}s...")
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(f"{request_label} failed.")

## 5. Chunk the section text

Each section is split into manageable chunks. The notebook caches LLM outputs per chunk so a crash or rate limit does not force you to restart from zero.

In [ ]:
# ============================================================
# CELL 5 — CHUNKING FOR STYLE EXTRACTION
# ============================================================

def split_text_into_chunks(text: str, max_chars: int = STYLE_CHUNK_MAX_CHARS) -> list[str]:
    paragraphs = re.split(r"\n\s*\n", text)
    chunks = []
    current = ""

    for para in paragraphs:
        para = para.strip()
        if not para:
            continue

        # If one paragraph is too long, split it roughly.
        if len(para) > max_chars:
            if current:
                chunks.append(current)
                current = ""
            for i in range(0, len(para), max_chars):
                chunks.append(para[i:i + max_chars])
            continue

        if len(current) + len(para) + 2 <= max_chars:
            current = current + "\n\n" + para if current else para
        else:
            if current:
                chunks.append(current)
            current = para

    if current:
        chunks.append(current)

    return chunks


section_chunks = {
    section: split_text_into_chunks(text)
    for section, text in section_texts.items()
}

chunk_rows = []
for section, chunks in section_chunks.items():
    for idx, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            "section": section,
            "chunk_index": idx,
            "char_count": len(chunk),
        })
    print(section, "chunks:", len(chunks), "chars:", sum(len(c) for c in chunks))

chunk_index_df = pd.DataFrame(chunk_rows)
chunk_index_path = INTERMEDIATE_DIR / "style_extraction_chunks.csv"
chunk_index_df.to_csv(chunk_index_path, index=False, encoding="utf-8-sig")

print("Saved chunk index:", chunk_index_path)
display(chunk_index_df.head(20))

## 6. Extract raw style notes from each chunk

This step extracts **abstract style notes only**. It explicitly excludes reference-company facts, names, targets, people, committees, numbers, awards and claims.

In [ ]:
# ============================================================
# CELL 6 — LLM STYLE NOTE EXTRACTION
# ============================================================

STYLE_CHUNK_SYSTEM_PROMPT = """
You are a senior sustainability reporting style analyst.

You extract reusable style, structure, formatting, PDF layout, and disclosure presentation patterns from a reference IFRS S1/S2 sustainability report.

Important:
- Do not copy sentences from the reference report.
- Do not extract company-specific facts, metrics, names, amounts, achievements, targets, people, committees, vendors, tools, locations, or claims as reusable content.
- Focus only on abstract writing style, structure, table behavior, figure behavior, PDF layout behavior, and disclosure presentation.
- The output will be used to guide report generation for a different company.
- Return valid JSON only.
""".strip()


def build_style_chunk_prompt(section_name: str, chunk_index: int, chunk_text: str) -> str:
    return f"""
Analyze this excerpt from the reference report section: {section_name}.

Extract only reusable style, structure and PDF layout patterns.

Return JSON with this schema:
{{
  "section_name": "{section_name}",
  "chunk_index": {chunk_index},
  "tone_patterns": [],
  "paragraph_patterns": [],
  "heading_patterns": [],
  "table_patterns": [],
  "figure_or_diagram_patterns": [],
  "pdf_layout_patterns": [],
  "disclosure_language_patterns": [],
  "evidence_presentation_patterns": [],
  "section_specific_observations": [],
  "things_to_avoid_copying": [],
  "content_specific_items_detected_and_excluded": []
}}

Rules:
- Do not quote the reference report.
- Do not include reference-company-specific names, facts, amounts, targets, awards, committees, people, locations, or numbers as style rules.
- Do not include any copied sentence.
- Use abstract descriptions only.

REFERENCE EXCERPT:
{chunk_text}
""".strip()


def cache_path_for_style_note(section_name: str, chunk_index: int) -> Path:
    slug = SECTION_SLUGS[section_name]
    return STYLE_NOTE_CACHE_DIR / f"{slug}_chunk_{chunk_index:03d}.json"


style_chunk_notes = {}

for section_name, chunks in section_chunks.items():
    style_chunk_notes[section_name] = []

    for idx, chunk in enumerate(chunks, start=1):
        cache_path = cache_path_for_style_note(section_name, idx)

        if cache_path.exists() and not FORCE_STYLE_EXTRACTION:
            with open(cache_path, "r", encoding="utf-8") as f:
                result = json.load(f)
            print(f"Loaded cached style notes: {section_name} chunk {idx}/{len(chunks)}")
        else:
            print(f"Extracting style notes: {section_name} chunk {idx}/{len(chunks)}")
            result = azure_chat_json(
                system_prompt=STYLE_CHUNK_SYSTEM_PROMPT,
                user_prompt=build_style_chunk_prompt(section_name, idx, chunk),
                request_label=f"Style notes {section_name} {idx}",
            )

            with open(cache_path, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

            time.sleep(STYLE_INTER_REQUEST_DELAY_SECONDS)

        style_chunk_notes[section_name].append(result)


# Save combined raw notes
raw_notes_path = INTERMEDIATE_DIR / "style_chunk_notes.json"

with open(raw_notes_path, "w", encoding="utf-8") as f:
    json.dump(style_chunk_notes, f, ensure_ascii=False, indent=2)

print("Saved combined notes:", raw_notes_path)

## 7. Consolidate raw notes into section style guides and section blueprints

This produces one style guide and one blueprint per target report section.

In [ ]:
# ============================================================
# CELL 7 — CONSOLIDATE SECTION STYLE GUIDES
# ============================================================

SECTION_CONSOLIDATION_SYSTEM_PROMPT = """
You are a senior sustainability reporting architect.

You convert raw style notes into a clean, reusable section style guide and section blueprint for IFRS S1/S2 reporting.

Important:
- Remove all reference-company-specific content.
- Remove all numbers, amounts, achievements, names, locations, people, committee names, tools, vendors and unique claims from the reference report.
- Keep only abstract, reusable style and structure rules.
- The output must be safe to use for generating a report for another company.
- Return valid JSON only.
""".strip()


def build_section_consolidation_prompt(section_name: str, notes: list[dict]) -> str:
    return f"""
Create a clean style guide and blueprint for the section: {section_name}.

Use the raw notes below, but remove all company-specific content.

Return JSON with this schema:
{{
  "section_name": "{section_name}",
  "section_style_guide": {{
    "purpose": "",
    "tone": "",
    "level_of_detail": "",
    "paragraph_style": "",
    "sentence_style": "",
    "preferred_disclosure_verbs": [],
    "preferred_evidence_style": "",
    "table_usage": "",
    "figure_usage": "",
    "pdf_layout_usage": "",
    "how_to_discuss_missing_data": "",
    "what_to_avoid": []
  }},
  "section_blueprint": {{
    "recommended_subsections": [],
    "recommended_order": [],
    "recommended_tables": [],
    "recommended_figures_or_diagrams": [],
    "recommended_pdf_blocks": [],
    "required_narrative_blocks": [],
    "optional_narrative_blocks": [],
    "output_structure_rules": []
  }},
  "section_specific_no_copying_rules": [],
  "quality_checks_for_generation": []
}}

Do not include:
- reference-company-specific names
- person names
- committee names unique to the reference company
- amounts
- percentages
- achievements
- awards
- exact copied wording

RAW STYLE NOTES:
{json.dumps(notes, ensure_ascii=False, indent=2)}
""".strip()


section_style_outputs = {}

for section_name, notes in style_chunk_notes.items():
    slug = SECTION_SLUGS[section_name]
    consolidated_path = INTERMEDIATE_DIR / f"{slug}_consolidated_style_output.json"

    if consolidated_path.exists() and not FORCE_STYLE_EXTRACTION:
        with open(consolidated_path, "r", encoding="utf-8") as f:
            result = json.load(f)
        print("Loaded cached consolidation:", section_name)
    else:
        print("Consolidating:", section_name)
        result = azure_chat_json(
            system_prompt=SECTION_CONSOLIDATION_SYSTEM_PROMPT,
            user_prompt=build_section_consolidation_prompt(section_name, notes),
            request_label=f"Consolidate {section_name}",
        )

        with open(consolidated_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        time.sleep(STYLE_INTER_REQUEST_DELAY_SECONDS)

    section_style_outputs[section_name] = result

    style_path = SECTION_STYLE_DIR / f"{slug}_style.json"
    blueprint_path = SECTION_BLUEPRINT_DIR / f"{slug}_blueprint.json"

    with open(style_path, "w", encoding="utf-8") as f:
        json.dump(result["section_style_guide"], f, ensure_ascii=False, indent=2)

    with open(blueprint_path, "w", encoding="utf-8") as f:
        json.dump(result["section_blueprint"], f, ensure_ascii=False, indent=2)

    print("Saved:", style_path)
    print("Saved:", blueprint_path)

## 8. Build the global style guide, table patterns and no-copying rules

The global guide controls the overall writing style. Section guides control each agent's local behavior.

In [ ]:
# ============================================================
# CELL 8 — GLOBAL STYLE GUIDE + TABLE PATTERNS + NO-COPY RULES
# ============================================================

GLOBAL_STYLE_SYSTEM_PROMPT = """
You are a senior sustainability reporting architect.

You create a global style guide from section-level style guides.

Important:
- Keep only abstract style, structure, formatting and PDF layout rules.
- Do not include reference-company-specific facts, numbers, names, achievements, targets, people, locations or copied wording.
- Do not ban the target company's own names/committees/policies/tools when they are explicitly supported by the target company payload. Ban only reference-company-specific content.
- Return valid JSON only.
""".strip()


def build_global_style_prompt(section_style_outputs: dict) -> str:
    return f"""
Create the final global style system for an IFRS S1/S2-aligned sustainability report that will be generated as a PDF.

Return JSON with this schema:
{{
  "global_style_guide": {{
    "report_voice": "",
    "tone": "",
    "point_of_view": "",
    "paragraph_rules": [],
    "sentence_rules": [],
    "disclosure_language_rules": [],
    "evidence_and_traceability_rules": [],
    "missing_data_language_rules": [],
    "formatting_rules": [],
    "do_not_do": []
  }},
  "table_patterns": {{
    "general_table_rules": [],
    "governance_tables": [],
    "strategy_tables": [],
    "risk_management_tables": [],
    "metrics_and_targets_tables": [],
    "recommended_columns_by_table_type": {{}}
  }},
  "no_copying_rules": [
    "Do not copy sentences from the reference report.",
    "Do not reuse reference-company-specific facts, names, amounts, awards, targets, people, committee names, locations, tools, vendors, images or claims.",
    "Use the reference only for abstract style, structure, formatting and PDF layout behavior.",
    "The generated report may include the target company's own names, committees, policies, systems, frameworks, tools, locations, targets and metrics only when explicitly supported by the target company payload or approved source data."
  ],
  "style_compliance_checks": []
}}

SECTION STYLE OUTPUTS:
{json.dumps(section_style_outputs, ensure_ascii=False, indent=2)}
""".strip()


global_style_cache_path = INTERMEDIATE_DIR / "global_style_result.json"

if global_style_cache_path.exists() and not FORCE_STYLE_EXTRACTION:
    with open(global_style_cache_path, "r", encoding="utf-8") as f:
        global_style_result = json.load(f)
    print("Loaded cached global style result.")
else:
    global_style_result = azure_chat_json(
        system_prompt=GLOBAL_STYLE_SYSTEM_PROMPT,
        user_prompt=build_global_style_prompt(section_style_outputs),
        request_label="Global style guide",
    )

    with open(global_style_cache_path, "w", encoding="utf-8") as f:
        json.dump(global_style_result, f, ensure_ascii=False, indent=2)


global_style_path = STYLE_OUTPUT_DIR / "global_style_guide.json"
table_patterns_path = TABLE_PATTERN_DIR / "table_patterns.json"
no_copying_path = LANGUAGE_RULES_DIR / "no_copying_rules.md"

with open(global_style_path, "w", encoding="utf-8") as f:
    json.dump(global_style_result["global_style_guide"], f, ensure_ascii=False, indent=2)

with open(table_patterns_path, "w", encoding="utf-8") as f:
    json.dump(global_style_result["table_patterns"], f, ensure_ascii=False, indent=2)

with open(no_copying_path, "w", encoding="utf-8") as f:
    f.write("# No-copying rules for report generation\n\n")
    for rule in global_style_result["no_copying_rules"]:
        f.write(f"- {rule}\n")

print("Saved:", global_style_path)
print("Saved:", table_patterns_path)
print("Saved:", no_copying_path)

## 9. Patch artifacts for PDF report generation

This fixes the over-strict company-name rule and adds a dedicated `layout_style_guide.json` for the final PDF report.

In [ ]:
# ============================================================
# CELL 9 — PATCH STYLE ARTIFACTS FOR PDF REPORT GENERATION
# Fixes:
# 1) Do not ban target-company names; ban only reference-company names.
# 2) Add PDF layout style guidance.
# 3) Strengthen no-copying rules for style/reference separation.
# ============================================================

global_style_path = STYLE_OUTPUT_DIR / "global_style_guide.json"
layout_style_path = STYLE_OUTPUT_DIR / "layout_style_guide.json"
no_copying_path = LANGUAGE_RULES_DIR / "no_copying_rules.md"

with open(global_style_path, "r", encoding="utf-8") as f:
    global_style = json.load(f)

# ------------------------------------------------------------
# 1. Patch strict/ambiguous no-copying rules
# ------------------------------------------------------------

old_rule = "Do not include company-specific names, people, committee titles, proprietary tool/framework names, or vendor/provider names."
new_rule = (
    "Do not include reference-company-specific names, people, committee titles, proprietary tools, vendor names, "
    "amounts, targets, awards, locations or claims from the style reference report. "
    "The generated report may include the target company’s own names, committees, policies, systems, frameworks, "
    "metrics and governance roles only when explicitly supported by the company payload or approved source data."
)

global_style["do_not_do"] = [
    new_rule if item == old_rule else item
    for item in global_style.get("do_not_do", [])
]

old_marketing_rule = "Do not use marketing language, advocacy framing, awards/leadership claims, or unsubstantiated effectiveness statements."
new_marketing_rule = (
    "Do not use marketing language, advocacy framing, awards/leadership claims, or effectiveness statements "
    "unless they are explicitly supported by target-company evidence and written neutrally."
)

global_style["do_not_do"] = [
    new_marketing_rule if item == old_marketing_rule else item
    for item in global_style.get("do_not_do", [])
]

# If the model phrased a similar broad ban, append a clarifying rule.
clarifying_rule = (
    "Reference-company-specific names, facts and design elements are forbidden; target-company-specific names, "
    "committees, tools, policies, frameworks, metrics and governance roles are allowed only when supported by the target payload."
)
if clarifying_rule not in global_style.get("do_not_do", []):
    global_style.setdefault("do_not_do", []).append(clarifying_rule)

# ------------------------------------------------------------
# 2. Add PDF-aware rules
# ------------------------------------------------------------

global_style.setdefault("formatting_rules", [])
global_style.setdefault("paragraph_rules", [])

pdf_formatting_rules = [
    "For PDF production, plan each major section as a report spread: section divider page, section overview, detailed narrative, tables/figures, and a short synthesis or cross-reference.",
    "Use section opener pages with section number, section title, and optional thematic image placeholder; avoid putting dense disclosure text on section divider pages.",
    "Use two-column or modular layouts for long narrative sections only when readability is preserved; keep compliance-critical text in a clear single-column flow when needed.",
    "Use callout boxes sparingly for key definitions, boundary notes, data limitations, or methodology notes.",
    "For large tables, design for PDF readability: concise cells, wrapped text, repeated headers, clear units, and explicit table captions."
]

for rule in pdf_formatting_rules:
    if rule not in global_style["formatting_rules"]:
        global_style["formatting_rules"].append(rule)

pdf_paragraph_rules = [
    "For PDF readability, alternate narrative blocks with tables, diagrams, or callouts in long sections such as Strategy and Metrics and Targets.",
    "Avoid pages made entirely of dense text; introduce visual anchors such as section subtitles, summary boxes, or compact tables where appropriate."
]

for rule in pdf_paragraph_rules:
    if rule not in global_style["paragraph_rules"]:
        global_style["paragraph_rules"].append(rule)

with open(global_style_path, "w", encoding="utf-8") as f:
    json.dump(global_style, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 3. Create layout_style_guide.json
# ------------------------------------------------------------

layout_style_guide = {
    "purpose": "PDF layout guidance derived from the reference report style, expressed as reusable design rules for a different IFRS S1/S2 report.",
    "core_principle": "Use the reference report as inspiration for professional PDF structure and visual hierarchy, not as a content or visual asset source.",
    "major_section_structure": {
        "section_divider_page": {
            "use": "Start every major section with a divider page.",
            "elements": [
                "large section number",
                "section title",
                "optional full-page or partial-page thematic image placeholder",
                "minimal text"
            ],
            "avoid": [
                "dense disclosure paragraphs",
                "tables",
                "company-specific imagery copied from the reference report"
            ]
        },
        "section_opening_page": {
            "use": "Follow the divider with a section opening page containing the section title and a short overview.",
            "elements": [
                "section title",
                "short purpose paragraph",
                "subsection navigation or key disclosure themes",
                "optional figure/table lead-in"
            ]
        },
        "detail_pages": {
            "use": "Use detail pages for narrative, processes, tables, figures and evidence-backed disclosure.",
            "elements": [
                "numbered subsections",
                "short paragraphs",
                "bullets for principles and process steps",
                "tables for metrics, risks, governance responsibilities and mappings",
                "figures for governance flow, value chain, risk lifecycle and financed-emissions scope"
            ]
        }
    },
    "visual_hierarchy": [
        "Use clear hierarchical numbering for major sections and subsections.",
        "Use section labels consistently in page headers or footers.",
        "Use captions for every figure and table.",
        "Separate overview, methodology, data limitations and results visually.",
        "Keep compliance-critical text readable before decorative design."
    ],
    "section_layout_rules": {
        "General Requirements": [
            "Use concise basis-of-preparation layout.",
            "Prefer short subsections for reporting period, boundaries, fair presentation, connected information, sources of guidance, compliance and materiality.",
            "Use a compact summary table only if it improves traceability."
        ],
        "Governance": [
            "Use diagrams or responsibility matrices to show oversight structure.",
            "Use role-based blocks: body/role, mandate, responsibility, reporting or escalation path.",
            "Use tables for board/committee responsibilities, skills, frequency and oversight topics when supported by payload."
        ],
        "Strategy": [
            "Use a more visual and narrative layout than other sections.",
            "Use value-chain diagrams, risk/opportunity matrices and time-horizon tables.",
            "Break long strategy narratives into thematic blocks with clear labels."
        ],
        "Risk Management": [
            "Use process-flow layouts for identify, assess, prioritise, monitor and integrate.",
            "Use tables for risk type, process, owner, control, monitoring mechanism and escalation.",
            "Separate general sustainability risk management from climate-specific risk management where applicable."
        ],
        "Metrics and Targets": [
            "Use table-first layout.",
            "Include metric name, definition, value, unit, source, boundary, target, status and related report section when available.",
            "Use methodology and limitation callouts for emissions, financed emissions and estimated metrics.",
            "Keep narrative commentary brief and directly linked to table content."
        ]
    },
    "pdf_readability_rules": [
        "Avoid long paragraphs wider than a normal report column.",
        "Use bullets for lists longer than three items.",
        "Avoid tables with paragraph-length cells unless no alternative exists.",
        "Use page breaks before major sections and before very large tables.",
        "Do not use figures as decoration only; every figure must support disclosure understanding.",
        "Use consistent formatting for data gaps, estimates and externally assured metrics."
    ],
    "no_copying_rules_for_layout": [
        "Do not reuse reference report images, icons, page compositions or distinctive brand design.",
        "Do not copy exact figure structure if it includes reference-company-specific governance or value-chain entities.",
        "Recreate only abstract layout patterns such as divider pages, numbered sections, responsibility matrices and metric tables."
    ]
}

with open(layout_style_path, "w", encoding="utf-8") as f:
    json.dump(layout_style_guide, f, ensure_ascii=False, indent=2)

# ------------------------------------------------------------
# 4. Rewrite no-copying rules
# ------------------------------------------------------------

no_copying_text = """# No-copying and style-source rules for report generation

The reference report is used only to extract reusable style, structure, formatting and PDF layout patterns.

## Allowed

- Use abstract writing style rules.
- Use abstract section structures.
- Use generic table and figure patterns.
- Use PDF layout principles such as section divider pages, numbered sections, captions and scannable tables.
- Use the target company's own committee names, policies, systems, frameworks, people, tools, locations, targets and metrics only when explicitly supported by the target company payload or approved source data.

## Forbidden

- Do not copy sentences from the reference report.
- Do not reuse the reference company's facts, names, amounts, awards, targets, people, committees, locations, tools, vendors, images or claims.
- Do not paste large portions of the reference report into generation prompts.
- Do not reuse reference report numbers as examples in generated disclosure.
- Do not imitate distinctive brand design, imagery or exact page compositions.
- Do not use unsupported marketing or leadership claims.

## Required behavior

- Style guides control tone, structure and formatting only.
- IFRS requirements control disclosure coverage.
- Company payloads control factual content.
- Unsupported disclosure requirements must be marked as data gaps rather than invented.
"""

with open(no_copying_path, "w", encoding="utf-8") as f:
    f.write(no_copying_text)

print("Patched:", global_style_path)
print("Created:", layout_style_path)
print("Updated:", no_copying_path)

## 10. Validate style artifacts against reference-content leakage

This checks that style artifacts do not contain obvious reference-company-specific names, people, committees, amounts, locations, or other copied factual content.

In [ ]:
# ============================================================
# CELL 10 — STYLE ARTIFACT VALIDATION
# ============================================================

FORBIDDEN_REFERENCE_TERMS = [
    "Emirates NBD",
    "DenizBank",
    "Emirates Islamic",
    "Vijay Bains",
    "Manoj Chawla",
    "Patrick Sullivan",
    "BNRESGC",
    "BRC",
    "GRC",
    "AED",
    "USD",
    "Dubai",
    "UAE",
    "MENA",
    "MENAT",
    "Sustainable Finance Forum",
    "Board Nomination, Remuneration and Environmental Social Governance Committee",
    "Burj",
    "KPMG Lower Gulf",
    "Microsoft Sustainability Manager",
    "Sustainalytics",
]

# This catches obvious leaked reference metrics, years, amounts, percentages.
# It may occasionally produce false positives if a generic style file contains an unavoidable standard name.
AMOUNT_OR_METRIC_PATTERN = re.compile(
    r"(\b\d+(\.\d+)?\s?%|\bUSD\b|\bAED\b|\bbillion\b|\bmillion\b|\b\d{4}\b)",
    flags=re.IGNORECASE
)


def read_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="replace")


def validate_artifact_file(path: Path) -> dict:
    text = read_text_file(path)

    forbidden_hits = [
        term for term in FORBIDDEN_REFERENCE_TERMS
        if term.lower() in text.lower()
    ]

    metric_hits = AMOUNT_OR_METRIC_PATTERN.findall(text)

    return {
        "file": str(path),
        "forbidden_reference_terms": forbidden_hits,
        "amount_or_metric_like_patterns": [m[0] for m in metric_hits[:20]],
        "has_copying_risk": bool(forbidden_hits or metric_hits),
    }


artifact_paths = list(STYLE_OUTPUT_DIR.rglob("*.json")) + list(STYLE_OUTPUT_DIR.rglob("*.md"))

validation_rows = [
    validate_artifact_file(path)
    for path in artifact_paths
    if "_intermediate" not in str(path)
]

validation_df = pd.DataFrame(validation_rows)

validation_path = STYLE_OUTPUT_DIR / "style_artifact_validation.csv"
validation_df.to_csv(validation_path, index=False, encoding="utf-8-sig")

display(validation_df)

risky = validation_df[validation_df["has_copying_risk"] == True]

if len(risky):
    print("WARNING: Some style artifacts may contain reference-specific terms or numbers.")
    print("Review these files manually before using them in generation:")
    display(risky)
else:
    print("Validation passed: no obvious reference-specific leakage detected.")

forbidden_terms_path = LANGUAGE_RULES_DIR / "forbidden_reference_terms.json"

with open(forbidden_terms_path, "w", encoding="utf-8") as f:
    json.dump(FORBIDDEN_REFERENCE_TERMS, f, ensure_ascii=False, indent=2)

print("Saved validation:", validation_path)
print("Saved forbidden terms:", forbidden_terms_path)

## 11. Create final style extraction audit summary

This documents the source, section ranges, outputs, and validation result.

In [ ]:
# ============================================================
# CELL 11 — STYLE EXTRACTION AUDIT SUMMARY
# ============================================================

summary_path = STYLE_OUTPUT_DIR / "style_extraction_audit_summary.md"

with open(summary_path, "w", encoding="utf-8") as f:
    f.write("# Style Extraction Audit Summary\n\n")

    f.write("## Source\n\n")
    f.write(f"- Reference report: `{REFERENCE_REPORT_PDF}`\n")
    f.write("- Purpose: extract reusable style, structure, layout and formatting patterns only.\n")
    f.write("- The reference report must not be used as a factual source during report generation.\n")
    f.write("- The final report target is PDF, so a dedicated PDF layout style guide is created.\n\n")

    f.write("## Section ranges used\n\n")
    for section, r in section_ranges.items():
        f.write(f"- {section}: pages {r['start_page']}–{r['end_page']}\n")

    f.write("\n## Artifacts created\n\n")
    f.write(f"- `{global_style_path}`\n")
    f.write(f"- `{layout_style_path}`\n")
    f.write(f"- `{table_patterns_path}`\n")
    f.write(f"- `{no_copying_path}`\n")
    f.write(f"- `{SECTION_STYLE_DIR}`\n")
    f.write(f"- `{SECTION_BLUEPRINT_DIR}`\n")
    f.write(f"- `{validation_path}`\n\n")

    f.write("## Validation\n\n")
    f.write(f"- Files checked: {len(validation_df)}\n")
    f.write(f"- Files with possible copying/content leakage risk: {len(risky)}\n\n")

    if len(risky):
        f.write("### Files requiring review\n\n")
        for _, row in risky.iterrows():
            f.write(f"- `{row['file']}`\n")
        f.write("\n")

    f.write("## Usage rule\n\n")
    f.write(
        "Generation agents should receive only the extracted style artifacts, "
        "not the original reference report text.\n\n"
    )

    f.write("## Required inputs for generation agents\n\n")
    f.write("Each section generation agent should receive:\n\n")
    f.write("1. Section-specific IFRS requirements.\n")
    f.write("2. Section-specific company payload.\n")
    f.write("3. `global_style_guide.json`.\n")
    f.write("4. `layout_style_guide.json`.\n")
    f.write("5. Relevant `section_style_guides/<section>_style.json`.\n")
    f.write("6. Relevant `section_blueprints/<section>_blueprint.json`.\n")
    f.write("7. `table_patterns/table_patterns.json`.\n")
    f.write("8. `language_rules/no_copying_rules.md`.\n")

print("Saved:", summary_path)
print(summary_path.read_text(encoding="utf-8"))

## 12. Final output index

Run this cell to see all final style artifacts that should be used by the report generation notebook.

In [ ]:
# ============================================================
# CELL 12 — OUTPUT INDEX
# ============================================================

final_artifacts = []
for path in sorted(STYLE_OUTPUT_DIR.rglob("*")):
    if path.is_file() and "_intermediate" not in str(path):
        final_artifacts.append({
            "artifact": str(path.relative_to(STYLE_OUTPUT_DIR)),
            "path": str(path),
            "size_kb": round(path.stat().st_size / 1024, 2),
        })

final_artifacts_df = pd.DataFrame(final_artifacts)
output_index_path = STYLE_OUTPUT_DIR / "style_artifacts_index.csv"
final_artifacts_df.to_csv(output_index_path, index=False, encoding="utf-8-sig")

print("Style system folder:", STYLE_OUTPUT_DIR)
print("Output index:", output_index_path)
display(final_artifacts_df)

## Expected final folder

After the notebook runs, your output should look like this:

```text
notebooks/gen_data/style/style_system/
  global_style_guide.json
  layout_style_guide.json
  style_artifact_validation.csv
  style_extraction_audit_summary.md
  style_artifacts_index.csv
  table_patterns/
    table_patterns.json
  language_rules/
    no_copying_rules.md
    forbidden_reference_terms.json
  section_style_guides/
    general_requirements_style.json
    governance_style.json
    strategy_style.json
    risk_management_style.json
    metrics_and_targets_style.json
  section_blueprints/
    general_requirements_blueprint.json
    governance_blueprint.json
    strategy_blueprint.json
    risk_management_blueprint.json
    metrics_and_targets_blueprint.json
```

Use these artifacts in generation. Do **not** pass the original reference report to the generation agents.